# PWKD Option 3 — Prune ResNet18 Directly

Implements **Pruning While Knowledge Distillation** (Wang et al., 2025) using:
- **Teacher**: frozen pretrained ResNet30 (RadImageNet weights)
- **Student**: a deep copy of the *same* pretrained ResNet30, fine-tuned
  with PWKD simultaneously pruning and distilling

This is a same-architecture setup, closely analogous to the paper's
EDSR-32-256 → EDSR-16-64 setup but keeping the architecture fixed and
instead driving channels to zero through the differentiable sparsity penalty.
The wavelet channel-projection step is skipped (teacher/student channels match).

Starting from pretrained weights means the student begins at the teacher's
F1 level (~0.4) and PWKD nudges it toward a smaller, slightly lower-F1 model —
a much more favourable trade-off than training from scratch.

Produces 5 compressed models at pruning ratios [10%, 25%, 50%, 70%, 90%],
saved to `trained_models/pwkd_self_r18_<ratio>/` and uploaded to HuggingFace.

**Run all cells top to bottom. Requires GPU.**

In [1]:
import os, copy, subprocess
import torch

target = 'CS6423_knowledge_distillation_project'
if not os.getcwd().endswith(target):
    import sys
    os.chdir(os.path.join(os.getcwd(), target))
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

print(f'Working dir: {os.getcwd()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

subprocess.run(['pip', 'install', 'PyWavelets', '--quiet'], check=True)

Working dir: /home/cor10/CS6423_knowledge_distillation_project
Device: cuda


CompletedProcess(args=['pip', 'install', 'PyWavelets', '--quiet'], returncode=0)

In [2]:
import pandas as pd
from modules.dataset_prepper import datasetPrepper

data_prep = datasetPrepper(
    dataframe_path='data/labels.csv',
    image_dir='data/test_images',
).prepare(compute_class_weights=True)

NUM_CLASSES = len(data_prep.class_names)
print(f'Classes: {NUM_CLASSES}')
print(f'Train batches: {len(data_prep.train_loader)} | Val batches: {len(data_prep.val_loader)}')

Classes: 61
Train batches: 249 | Val batches: 50


In [3]:
from modules.imagenet_loader import ImagenetLoader
from modules.evaluate_model import ModelEvaluator

loader = ImagenetLoader()

# Load pretrained ResNet18 — this serves as both the frozen teacher
# and the starting point for each student deep copy
resnet18_pretrained = loader.load_radimagenet_resnet18(
    weights_path='trained_models/resnet18_baseline_gpu_new/resnet18_baseline_gpu_new.pth',
    load_type='load'
)
resnet18_pretrained = resnet18_pretrained.to(device)

evaluator = ModelEvaluator(
    data_loader=data_prep.val_loader,
    class_names=data_prep.class_names,
    device=str(device),
)

baseline_metrics = evaluator.evaluate_single(resnet18_pretrained, 'ResNet18_baseline')
BASELINE_PARAMS  = baseline_metrics['total_parameters']
print(f'ResNet18 baseline F1:    {baseline_metrics["f1_macro"]:.4f}')
print(f'ResNet18 baseline params: {BASELINE_PARAMS:,}')


Warming up ResNet18_baseline...
Running inference...
ResNet18 baseline F1:    0.4054
ResNet18 baseline params: 11,207,805


In [ ]:
import torch.nn as nn
import copy
from modules.model_trainer import modelTrainer
from modules.evaluate_model import ModelEvaluator
from pwkd.pwkd import PWKDLoss, make_aux_fn, finalise_student

RESNET18_CHANNELS = {'layer2': 128, 'layer3': 256, 'layer4': 512}

PRUNING_RATIOS = [0, 0.05, 0.10, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.50, 0.70, 0.90]
NUM_EPOCHS     = 10
KD_TEMP        = 4.0
LAM            = 0.2
SPARSE_WEIGHT  = 5e-4
learn_rate     = 1e-4

evaluator = ModelEvaluator(
    data_loader=data_prep.val_loader,
    class_names=data_prep.class_names,
    device=str(device),
)

teacher = copy.deepcopy(resnet18_pretrained).eval()
for p in teacher.parameters():
    p.requires_grad = False

pwkd_metrics = []

print(f'ResNet18 baseline F1:    {baseline_metrics["f1_macro"]:.4f}')
print(f'ResNet18 baseline params: {BASELINE_PARAMS:,}')


for ratio in PRUNING_RATIOS:
    label = f'pwkd_self_r18_r{int(ratio * 100)}'
    print(f'\n{"="*60}')
    print(f'  PWKD Option 3 (ResNet18) — pruning ratio {ratio:.0%}')
    print(f'{"="*60}')

    student = copy.deepcopy(resnet18_pretrained).to(device)
    for p in student.parameters():
        p.requires_grad = True
        
    # reset teacher each experiment
    teacher_copy = copy.deepcopy(teacher).to(device)
    teacher_copy.eval()

    pwkd_loss = PWKDLoss(
        student          = student,
        teacher          = teacher_copy,
        pruning_ratio    = ratio,
        teacher_channels = RESNET18_CHANNELS,
        student_channels = RESNET18_CHANNELS,
        class_weights    = data_prep.class_weights.to(device)
                           if data_prep.class_weights is not None else None,
        lam              = LAM,
        kd_temp          = KD_TEMP,
        sparse_weight    = 0 if ratio == 0 else SPARSE_WEIGHT,
    ).to(device)

    trainer = modelTrainer(
        model      = student,
        data_prep  = data_prep,
        device     = device,
        # learn_rate = 2e-4,
        learn_rate = learn_rate,
        num_epochs = NUM_EPOCHS,
        model_name = label,
    )
    trainer.loss_fn   = pwkd_loss
    trainer.optimizer = torch.optim.AdamW(
        list(student.parameters()) + list(pwkd_loss.parameters()),
        lr=5e-5 if ratio == 0 else learn_rate,
        weight_decay=1e-4,
    )
    trainer.create_classnum_to_label_map(data_prep.class_names)
    trainer.train_all(save_as_object=True, aux_forward_fn=make_aux_fn(teacher_copy), pwkd=True)
    
    # Reload best checkpoint before finalising
    best_ckpt = os.path.join('trained_models', label, f'{label}_full.pth')
    student = torch.load(best_ckpt, map_location=device, weights_only=False)['model']
    student = student.to(device)

    student = finalise_student(student, pwkd_loss, data_prep.train_loader, device)

    import os
    save_dir = os.path.join('trained_models', label)
    os.makedirs(save_dir, exist_ok=True)
    torch.save({'model': student, 'epoch': NUM_EPOCHS},
               os.path.join(save_dir, f'{label}_full.pth'))
    print(f'Saved → {save_dir}/{label}_full.pth')

    student.eval()
    metrics = evaluator.evaluate_single(student, label)

    pwkd_metrics.append({
        'Pruning Ratio':      f'{int(ratio*100)}%',
        'Size Reduction (%)': round((BASELINE_PARAMS - metrics['total_parameters'])
                                    / BASELINE_PARAMS * 100, 2),
        'F1 Score':           round(metrics['f1_macro'],     4),
        'Size (MB)':          round(metrics['model_size_mb'], 1),
        'Latency (ms)':       round(metrics['avg_latency_ms'], 2),
    })
    print(f'  ratio: {ratio:.0%} | F1: {metrics["f1_macro"]:.4f} | '
          f'params: {metrics["total_parameters"]:,} | '
          f'latency: {metrics["avg_latency_ms"]:.2f}ms')

summary_df = pd.DataFrame(pwkd_metrics).set_index('Pruning Ratio')
print('\nPWKD Option 3 (ResNet18) — Results')
display(summary_df)


ResNet18 baseline F1:    0.4054
ResNet18 baseline params: 11,207,805

  PWKD Option 3 (ResNet18) — pruning ratio 0%


Validating: 100%|██████████| 50/50 [00:03<00:00, 16.49batch/s]



Epoch 1/10
Train Loss: 0.6860 | Train F1: 0.6585
Val Loss: 1.4045 | Val F1: 0.4166
Epoch Time: 30.81s



Validating: 100%|██████████| 50/50 [00:02<00:00, 18.04batch/s]



Epoch 2/10
Train Loss: 0.4782 | Train F1: 0.7310
Val Loss: 1.3502 | Val F1: 0.4260
Epoch Time: 31.14s



...


Epoch 9/10
Train Loss: 0.4326 | Train F1: 0.8337
Val Loss: 1.3494 | Val F1: 0.4361
Epoch Time: 35.43s



Validating: 100%|██████████| 50/50 [00:02<00:00, 18.13batch/s]


Epoch 10/10
Train Loss: 0.4355 | Train F1: 0.8430
Val Loss: 1.3865 | Val F1: 0.4389
Epoch Time: 35.48s



Finalised: 1724/1920 conv1 channels zeroed (89.8%)
Saved → trained_models/pwkd_self_r18_r90/pwkd_self_r18_r90_full.pth

Warming up pwkd_self_r18_r90...
Running inference...
  ratio: 90% | F1: 0.0001 | params: 1,339,197 | latency: 0.39ms

PWKD Option 3 (ResNet18) — Results


,Size Reduction (%),F1 Score,Size (MB),Latency (ms)
Pruning Ratio,,,,
0%,0.00,0.4644,42.8,0.39
5%,4.74,0.3996,42.8,0.39
10%,9.69,0.3776,42.8,0.38
15%,14.54,0.3465,42.8,0.38
20%,19.49,0.2960,42.8,0.39
25%,24.50,0.2204,42.8,0.39
30%,29.24,0.0838,42.8,0.39
35%,34.20,0.0423,42.8,0.39
40%,39.04,0.0296,42.8,0.39


In [ ]:
print('\nPWKD Option 3 (ResNet18) — Results')
display(summary_df)

# this is learn_rate = 1e-4


PWKD Option 3 (ResNet18) — Results


,Size Reduction (%),F1 Score,Size (MB),Latency (ms)
Pruning Ratio,,,,
0%,0.00,0.4644,42.8,0.39
5%,4.74,0.3996,42.8,0.39
10%,9.69,0.3776,42.8,0.38
15%,14.54,0.3465,42.8,0.38
20%,19.49,0.2960,42.8,0.39
25%,24.50,0.2204,42.8,0.39
30%,29.24,0.0838,42.8,0.39
35%,34.20,0.0423,42.8,0.39
40%,39.04,0.0296,42.8,0.39
